# VWAP Reversion — Gate 1 measurement

Hypothesis: `research/hypotheses/vwap_reversion.md`

Two runs on 2018-01-02 → 2022-12-30, shared entries, EOD-flat at 15:45 NY.

- **Run 1 — VWAP-Touch + Time-Stop** (`exit_mode='signal'`): limit TP on VWAP touch, time-stop at 6 bars (90 min), EOD-flat. No adverse-price stop by design (Cantarutti).
- **Run 2 — Default ATR wrapper** (`exit_mode='atr'`, 1.5× / 2.0×): symmetric volatility baseline on identical entries.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from engine.data_loader import load_csv
from engine.features import build_features, build_vwap_features
from engine.gate1 import gate1_evaluate, print_gate1

RNG_SEED = 42
BOOTSTRAP_ITERATIONS = 10_000

In [ ]:
df_full = load_csv(str(PROJECT_ROOT / 'data' / 'nq_15m_data.csv'), session_filter=True)
df = df_full.loc['2018-01-02':'2022-12-30'].copy()
print(f'Training slice: {df.index.min()}  →  {df.index.max()}')
print(f'Bars: {len(df):,}')

## Build features and signals

Locked Gate 1 base case:
- `entry_z = -2.0`
- entry window: 10:30 ≤ t ≤ 15:00 NY
- `max_hold_bars = 6` (Run 1 only — Run 2 uses ATR stop)
- direction: LONG only

In [ ]:
ENTRY_Z = -2.0
MAX_HOLD_BARS = 6
ENTRY_START = pd.Timestamp('10:30').time()
ENTRY_END = pd.Timestamp('15:00').time()

sigs = build_features(df)
sigs = build_vwap_features(sigs)

time_vals = sigs.index.time
in_window = (time_vals >= ENTRY_START) & (time_vals <= ENTRY_END)

entry_mask = (sigs['signal_zscore'] <= ENTRY_Z) & in_window
sigs['signal'] = entry_mask.fillna(False).astype(np.int8)

sigs['exit_tp_price'] = sigs['target_price']
sigs['exit_signal_stop'] = False  # Run 1 has no regime stop — time-stop only

print(f'Signal-eligible bars: {int(sigs["signal"].sum()):,}')

## Run 1 — VWAP-Touch + Time-Stop

In [ ]:
run1_overrides = {
    'exit_mode': 'signal',
    'max_concurrent_trades': 1,
    'max_bars_in_trade': MAX_HOLD_BARS,
}
g1_run1 = gate1_evaluate(sigs, wrapper_overrides=run1_overrides)
print_gate1(g1_run1)

## Run 2 — Default ATR wrapper

In [ ]:
run2_overrides = {
    'exit_mode': 'atr',
    'max_concurrent_trades': 1,
}
g1_run2 = gate1_evaluate(sigs, wrapper_overrides=run2_overrides)
print_gate1(g1_run2)

## Supplementary metrics + exit-reason breakdown

In [ ]:
def trade_pnls(result):
    return np.array([t.pnl for t in result.backtest.trades]) if result.backtest.trades else np.array([])

def supplementary(pnls, label):
    n = len(pnls)
    if n == 0:
        print(f'{label}: no trades'); return
    wins, losses = pnls[pnls > 0], pnls[pnls < 0]
    gp, gl = wins.sum(), losses.sum()
    wr = len(wins) / n
    exp_ = pnls.mean()
    sd = pnls.std(ddof=1) if n > 1 else 0.0
    t_stat = exp_ / (sd / np.sqrt(n)) if sd > 0 else float('nan')
    rng = np.random.default_rng(RNG_SEED)
    boot = rng.choice(pnls, size=(BOOTSTRAP_ITERATIONS, n), replace=True).mean(axis=1)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    print(f'{label}')
    print(f'  n trades          {n}')
    print(f'  win rate          {wr:.1%}')
    print(f'  gross profit      ${gp:,.2f}')
    print(f'  gross loss        ${gl:,.2f}')
    print(f'  net P&L           ${gp + gl:,.2f}')
    print(f'  expectancy/trade  ${exp_:,.2f}')
    print(f'  95% boot CI       [${lo:,.2f}, ${hi:,.2f}]')
    print(f'  t-statistic       {t_stat:.3f}')
    print()

def exit_breakdown(result, label):
    trades = result.backtest.trades
    if not trades:
        print(f'{label}: no trades'); return
    rs = pd.Series([t.exit_reason for t in trades]).value_counts()
    print(f'{label} — exit reasons: {dict(rs)}')

supplementary(trade_pnls(g1_run1), 'Run 1 — VWAP-Touch + Time-Stop')
supplementary(trade_pnls(g1_run2), 'Run 2 — Default ATR wrapper')
exit_breakdown(g1_run1, 'Run 1')
exit_breakdown(g1_run2, 'Run 2')

## Equity curves

MTM equity (bar-level, includes unrealised P&L) plus drawdown. Closed-trade equity is shown separately below.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax = axes[0]
(g1_run1.backtest.equity_mtm - 100_000).plot(ax=ax, label='Run 1 — VWAP-Touch + Time-Stop', color='C3', linewidth=0.9)
(g1_run2.backtest.equity_mtm - 100_000).plot(ax=ax, label='Run 2 — Default ATR', color='C0', linewidth=0.9)
ax.axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.5)
ax.set_ylabel('Cumulative net P&L ($)')
ax.set_title('VWAP Reversion — Gate 1 equity curves (training slice 2018–2022)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

ax = axes[1]
g1_run1.backtest.drawdown_series.plot(ax=ax, label='Run 1 DD %', color='C3', linewidth=0.9)
g1_run2.backtest.drawdown_series.plot(ax=ax, label='Run 2 DD %', color='C0', linewidth=0.9)
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Closed-trade equity (steps on exits only — no open-position MTM)
fig, ax = plt.subplots(figsize=(13, 4))
(g1_run1.backtest.equity_closed - 100_000).plot(ax=ax, label='Run 1 closed equity', color='C3', linewidth=0.9)
(g1_run2.backtest.equity_closed - 100_000).plot(ax=ax, label='Run 2 closed equity', color='C0', linewidth=0.9)
ax.axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.5)
ax.set_ylabel('Realised P&L ($)')
ax.set_title('Closed-trade equity (no open-position MTM)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()